In [96]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [72]:
#length of dataset
with open('tinyshakespeare.txt', 'r', encoding = 'utf-8') as f:
    text = f.read()
print('length of dataset in chars:', len(text))

length of dataset in chars: 1115394


In [73]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [74]:
chars = sorted(list(set(text)))
vocab_size = (len(chars))
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [75]:
stoi = { ch:i for i,ch in enumerate(chars)}
itos = { i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] #returns ints from strings
decode = lambda l: ''.join([itos[c] for c in l]) #returns strings from ints

print(encode('bleep'))
print(decode(encode('bleep')))

[40, 50, 43, 43, 54]
bleep


In [76]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [70]:
print(decode(data[:100].tolist()))

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [54]:
#set up train, validate data splits
n = int(len(data)*0.9)

train_data = data[:n]
val_data = data[n:]

In [94]:
torch.manual_seed(1337)
block_size = 8
batch_size = 4
def get_batch(split):
    data = train_data if split=='train' else val_data
    #get the ix randomly
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('outputs:')
print(yb.shape)
print(yb)
print('----')

for b in range(batch_size):
    for t in range (block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when context is {context.tolist()} the target is {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
outputs:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when context is [24] the target is 43
when context is [24, 43] the target is 58
when context is [24, 43, 58] the target is 5
when context is [24, 43, 58, 5] the target is 57
when context is [24, 43, 58, 5, 57] the target is 1
when context is [24, 43, 58, 5, 57, 1] the target is 46
when context is [24, 43, 58, 5, 57, 1, 46] the target is 43
when context is [24, 43, 58, 5, 57, 1, 46, 43] the target is 39
when context is [44] the target is 53
when context is [44, 53] the target is 56
when context is [44, 53, 56] the target is 1
when context is [44, 53, 56, 1] the target is 58
when context is [44, 5

In [109]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        #each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets == None:
            loss = None
        else:
            B, T, C= logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            
            loss = F.cross_entropy(logits, targets) #cross entropy expects (B, C) shape
        
        return logits, loss
        
    def generate(self, idx, max_new_tokens):
        #initial context
        context = []
        for _ in range(max_new_tokens):
            logits, loss = self(idx) # get predictions
            logits = logits[:, -1, :] #focus only on the last timestep 

            probs = F.softmax(logits, dim=-1) #use softmax to get probabilities

            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1) one prediction for each batch item

            idx = torch.cat((idx, idx_next), dim=1) # (B, t+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

idx = torch.zeros((1,1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.1236, grad_fn=<NllLossBackward0>)

KBYqcdK:gSSUF3SpzPo,pkmqI&ZjKb;ef
LoYmRnOH,K,:?zlSxxxsDt!szNmHK,PXyTeMHKs SSNyWNkBEL!jEnPQjiNBJfFQbL


In [118]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [119]:
batch_size = 32
for steps in range(1000):

    xb, yb = get_batch('train')

    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if steps % 100 == 0:
        print(loss.item())

2.4152283668518066
2.472999334335327
2.3485465049743652
2.603292942047119
2.4732718467712402
2.40370774269104
2.3753933906555176
2.348057985305786
2.4933977127075195
2.5765466690063477


In [121]:
print(decode(m.generate(idx, max_new_tokens=300)[0].tolist()))


CHENatoforss borr he w.
O:
A othe bre fod measo putrcigmu wh, tnt he verus,
GRINofafon t br gighy CE saved pus, qutha, rso sir d,
SI tsello, g'ecadod s s, s bells br:
Hefore.
Tad rathofe
SINI' tssplllo pset ims t wn de att we wingis
boues balles bowo--undwad k.
KENCE:
Awhowivearancour ald he th atod


In [130]:
torch.manual_seed(1337)
a = torch.tril(torch.ones(3,3))
a = a/torch.sum(a, 1, keepdim=True)
b = torch.randint(0, 10, (3,2)).float()
c = a@b
print('a=')
print(a)
print('b=')
print(b)
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b=
tensor([[5., 7.],
        [2., 0.],
        [5., 3.]])
c=
tensor([[5.0000, 7.0000],
        [3.5000, 3.5000],
        [4.0000, 3.3333]])


In [154]:
#v4 : SELF-ATTENTION!!!!
B,T,C = 4, 8, 32 #batch, time, channels
x = torch.randn(B,T,C)

#single attention head
head_size = 16
key = nn.Linear(C, head_size, bias=False)     #the topics I have to share
query = nn.Linear(C, head_size, bias=False)   #the topics what I'm looking for
value = nn.Linear(C, head_size, bias=False)   #the content I share

k = key(x)    # (B,T,C) @ (C, 16) ---> (B,T, 16) #key results
q = query(x)  # (B,T,C) @ (C, 16) ---> (B,T, 16) #value results

#dot product the queries with the keys
wei = q @ k.transpose(-2, -1) * head_size**-0.5 # (B,T,16) @ (B,16,T) ----> (B,T,T)

tril = torch.tril(torch.ones(8,8)) #triangle of zeroes in upper right
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x) # (B,T,C) @ (C, 16) ---> (B,T, 16)
out = wei@v # (B,T,T) @ (B,T, 16)  ---> (B,T, 16)

out.shape

torch.Size([4, 8, 16])

In [153]:
wei

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4258, 0.5742, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4186, 0.2718, 0.3096, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2959, 0.1392, 0.3016, 0.2634, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2648, 0.1535, 0.2404, 0.1374, 0.2039, 0.0000, 0.0000, 0.0000],
         [0.1480, 0.1323, 0.1930, 0.1619, 0.1686, 0.1962, 0.0000, 0.0000],
         [0.1386, 0.1177, 0.1921, 0.1178, 0.1213, 0.1720, 0.1405, 0.0000],
         [0.0919, 0.1749, 0.1074, 0.1125, 0.1162, 0.1330, 0.1613, 0.1029]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4007, 0.5993, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3017, 0.3595, 0.3388, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1527, 0.3146, 0.2824, 0.2503, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1764, 0.1437, 0.2384, 0.1819, 0.2596, 0.0000, 0.0000, 0.0000],
         [0.2271, 0.178